In [ ]:
import sys
sys.path.insert(0, "..")

import os
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from itertools import combinations
import seaborn.objects as so
import textwrap

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

from run_llm_eval import parse_choice

#DATA_DIR = Path("../outputs_nemo")
DATA_DIR = Path("../outputs_2026_02_27-google")

# 1) Discover files
pattern = re.compile(r"predictions.*\.csv$", re.IGNORECASE)
csv_files = [p for p in DATA_DIR.iterdir() if pattern.search(p.name)]


In [ ]:
def model_and_role_from_filename(path: Path):
    # e.g., predictions_openai_gpt-4o.csv -> openai_gpt-4o
    name = path.stem
    name = re.sub(r"^predictions?_?","",name, flags=re.IGNORECASE)
    name = re.sub(r"^[\-\._]+","", name)
    role = name.split("_")[-1]
    name = name.rsplit("_", 1)[0]
    return name, role

In [ ]:
# long dataformat for seaborn
dataframes = []

for fp in csv_files:
    df = pd.read_csv(fp)
    model_id, role = model_and_role_from_filename(fp)
    print(model_id, role)
    df.pred_choice = df["raw_reply"].apply(parse_choice)
    df = df[["ID", "pred_choice"]].rename(columns={"pred_choice": "value", "ID": "question_id"})
    df["role"] = role
    df["model"] = model_id
    
    dataframes.append(df)

results_df = pd.concat(dataframes, axis=0)
results_df.reset_index(drop=True, inplace=True)

In [ ]:
questions = pd.read_csv(fp)[["ID", "Question"]].set_index("ID")
questions["wrapped"] = questions["Question"].apply(lambda x: "\n".join(textwrap.wrap(x, width=40)))
questions.head()

In [ ]:
# export to excel
from collections import defaultdict
dataframes = defaultdict(list)
dataframes_to_write = {}

for fp in csv_files:
    df = pd.read_csv(fp)
    temp_q = df[["ID", "Question"]].set_index("ID")
    model_id, role = model_and_role_from_filename(fp)
    df.pred_choice = df["raw_reply"].apply(parse_choice)
    df = df[["ID", "pred_choice"]].set_index("ID").rename(columns={"pred_choice": model_id})
    dataframes[role].append(df)

for role, dfs in dataframes.items():
    print(role)
    combined_df = pd.concat([temp_q] + dfs, axis=1)
    dataframes_to_write[role] = combined_df

with pd.ExcelWriter(f"{DATA_DIR}/all_results.xlsx") as writer:
    for sheet_name, dataframe in dataframes_to_write.items():
        dataframe.to_excel(writer, sheet_name=sheet_name, index=True)

## Line Charts

In [ ]:
path = Path(f"{DATA_DIR}/figures")
os.makedirs(path, exist_ok=True)

model_order = sorted(results_df["model"].unique())
#role_order = ['Keine', 'Assistant', 'Sustainability Manager', 'NGO', 'CDU', 'CFO', 'Startup']
#role_order = ['Keine', 'Syphocancy-pro', 'Syphocancy-gegen']
role_order = ['Keine', 'Zweipersonenhaushalt', 'Zweipersonenhaushalt-stadt', 'Zweipersonenhaushalt-land', 'Vierpersonenhaushalt',
              "Zweipersonenhaushalt-usa", "Zweipersonenhaushalt-china", "Zweipersonenhaushalt-nigeria"]

zahl_questions = [
    "Verhalten10",
    "Verhalten11",
    "Verhalten12",
    "Verhalten13",
    "Verhalten14",
    "Verhalten15",
    "Verhalten16",
    "Verhalten17",
    "Verhalten18",
    "Verhalten19",
    "Verhalten20",
    "Verhalten21",
    "Verhalten22",
    "Verhalten23",
    "Verhalten24",
    "WTP1",
    "WTP2",
    "WTP3",
    "WTP4",
    "WTP5",
    "WTP11"
]

for q in questions.index:
    results_q = results_df[(results_df["question_id"] == q) & (results_df["role"]=="Keine")]
    if q in zahl_questions:
        p = (so.Plot(data=results_q, x="value", y="model")
            .label(title=questions.loc[q, "wrapped"], x="", y="", title_loc='left')
            .theme({"axes.facecolor": "white", "figure.facecolor": "white"})
            .scale(y=so.Nominal(order=model_order))
            .add(so.Dot(), so.Jitter(x=.2, y=.2))
            .add(so.Line(marker="o", linestyle="-"), so.Jitter(y=.2))
        )
    else:
        p = (so.Plot(data=results_q, x="value", y="model").limit(x=(0.5, 4.5))
            .label(title=questions.loc[q, "wrapped"], x="", y="", title_loc='left')
            .theme({"axes.facecolor": "white", "figure.facecolor": "white"})
            .scale(y=so.Nominal(order=model_order))
            .add(so.Dot(), so.Jitter(x=.2, y=.2))
            .add(so.Line(marker="o", linestyle="-"), so.Jitter(y=.2))
        )
    p.save(path / f"{q}.png", dpi=300, bbox_inches='tight')

In [ ]:
path = Path(f"{DATA_DIR}/figures_role")
os.makedirs(path, exist_ok=True)

for q in questions.index:
    results_q = results_df[results_df["question_id"] == q]
    if q in zahl_questions:
        p = (so.Plot(data=results_q, x="value", y="model", color="role")
            .label(title=questions.loc[q, "wrapped"], x="", y="", title_loc='left')
            .theme({"axes.facecolor": "white", "figure.facecolor": "white"})
            .scale(y=so.Nominal(order=model_order))
            .scale(color=so.Nominal(order=role_order))
            .add(so.Dot(), so.Jitter(x=.2, y=.2))
            .add(so.Line(marker="o", linestyle="-"), so.Jitter(y=.2))
        )
    else:
        p = (so.Plot(data=results_q, x="value", y="model", color="role").limit(x=(0.5, 4.5))
            .label(title=questions.loc[q, "wrapped"], x="", y="", title_loc='left')
            .theme({"axes.facecolor": "white", "figure.facecolor": "white"})
            .scale(y=so.Nominal(order=model_order))
            .scale(color=so.Nominal(order=role_order))
            .add(so.Dot(), so.Jitter(x=.2, y=.2))
            .add(so.Line(marker="o", linestyle="-"), so.Jitter(y=.2))
        )
    p.save(path / f"{q}.png", dpi=300, bbox_inches='tight')

## Clustering

In [ ]:
# Load the dataset
df = pd.read_excel(f"{DATA_DIR}/all_results.xlsx", sheet_name='Keine')

# Data Cleaning and Preprocessing
# Set ID as index and drop non-numeric description columns
df_analysis = df.set_index('ID').drop(columns=['Question'])

# Convert columns to numeric, coercing errors to NaN (in case of text responses)
for col in df_analysis.columns:
    df_analysis[col] = pd.to_numeric(df_analysis[col], errors='coerce')

df_analysis = df_analysis.dropna(axis=1, how='all')
df_analysis = df_analysis.dropna(axis=0, how='all')
df_analysis = df_analysis.fillna(df_analysis.median())

# Transpose: We want to cluster MODELS (columns), not QUESTIONS (rows).
df_transposed = df_analysis.T


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Pearson (Linear)
sns.heatmap(df_transposed.T.corr(method='pearson'), 
            annot=True, cmap='coolwarm', fmt=".2f", ax=axes[0])
axes[0].set_title('Pearson (Empfindlich für absolute Werte)')

# Spearman (Rang)
sns.heatmap(df_transposed.T.corr(method='spearman'), 
            annot=True, cmap='coolwarm', fmt=".2f", ax=axes[1])
axes[1].set_title('Spearman (Robust / Muster-basiert)')
plt.tight_layout()
plt.savefig(DATA_DIR / f"model_correlation_heatmaps.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# A) Standard PCA (Werte-basiert)
scaler = StandardScaler()
coords_std = PCA(n_components=2).fit_transform(scaler.fit_transform(df_transposed))
sns.scatterplot(x=coords_std[:,0], y=coords_std[:,1], s=100, ax=axes[0])
for i, txt in enumerate(df_transposed.index):
    axes[0].annotate(txt, (coords_std[i,0]+0.1, coords_std[i,1]))
axes[0].set_title('PCA auf Rohdaten (Werte-basiert)')
axes[0].grid(True)

# B) Rang PCA (Muster-basiert)
# Trick: Wir wandeln die Daten erst in Ränge um!
df_ranked = df_transposed.rank(axis=1) 
coords_rank = PCA(n_components=2).fit_transform(scaler.fit_transform(df_ranked))
sns.scatterplot(x=coords_rank[:,0], y=coords_rank[:,1], s=100, ax=axes[1])
for i, txt in enumerate(df_transposed.index):
    axes[1].annotate(txt, (coords_rank[i,0]+0.1, coords_rank[i,1]))
axes[1].set_title('PCA auf Rang-Daten (Struktur-basiert)')
axes[1].grid(True)

plt.tight_layout()
plt.savefig(DATA_DIR / f"model_pca.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10)) # Breite Bildfläche

# --- Linker Plot: Standard Scaler (Werte) ---
scaler = StandardScaler()
# Wichtig: fit_transform erwartet (Samples, Features). Wir haben (Modelle, Fragen).
df_scaled = scaler.fit_transform(df_transposed)

# Ward-Methode minimiert Varianz innerhalb der Cluster
Z_std = sch.linkage(df_scaled, method='ward')

sch.dendrogram(
    Z_std,
    labels=df_transposed.index, # Modellnamen
    orientation='left',
    leaf_font_size=11,
    ax=axes[0] # Zeichne in den linken Slot
)
axes[0].set_title('A) Basiert auf absoluten Werten (Standard Scaler)')
axes[0].set_xlabel('Distanz')

# --- Rechter Plot: Spearman (Muster) ---
# Berechne Korrelation zwischen MODELLEN (deshalb .T.corr)
corr_matrix = df_transposed.T.corr(method='spearman')
distance_matrix = 1 - corr_matrix
condensed_distance = squareform(distance_matrix)

# Average-Linkage ist Standard für Korrelations-Distanzen
Z_spear = sch.linkage(condensed_distance, method='average')

sch.dendrogram(
    Z_spear,
    labels=df_transposed.index,
    orientation='left',
    leaf_font_size=11,
    ax=axes[1] # Zeichne in den rechten Slot
)
axes[1].set_title('B) Basiert auf Antwort-Mustern (Spearman)')
axes[1].set_xlabel('Distanz (1 - Korrelation)')

plt.tight_layout()
plt.savefig(DATA_DIR / f"model_correlation_dendrograms.png", dpi=300, bbox_inches='tight')
plt.show()